In [1]:
#pip install pgmpy networkx matplotlib

In [2]:
import pandas as pd

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import (
    MaximumLikelihoodEstimator,
    BayesianEstimator,
    HillClimbSearch,
)
from pgmpy.inference import VariableElimination


DATASET_PATH = "../data/processed/dataset.csv"

BN_COLUMNS = [
    "ht_form",
    "at_form",
    "joint_attack",
    "h2h_dominance",
    "odds_favorite",
    "result",
]


def load_bn_data(path=DATASET_PATH):
    df = pd.read_csv(path, parse_dates=["date"])
    df = df.sort_values("date").reset_index(drop=True)

    data = df[BN_COLUMNS].copy()

    for col in data.columns:
        data[col] = data[col].astype(str)

    return data


def temporal_train_test_split(data, train_size=0.8):
    split_idx = int(len(data) * train_size)
    train_data = data.iloc[:split_idx].copy()
    test_data = data.iloc[split_idx:].copy()
    return train_data, test_data


data = load_bn_data()
train_data, test_data = temporal_train_test_split(data)

state_names = {
    col: sorted(data[col].unique().tolist())
    for col in BN_COLUMNS
}

print(train_data.head())
print(state_names)

c:\Users\duart\OneDrive\Ambiente de Trabalho\Classes\semestre2\IAP\project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\duart\OneDrive\Ambiente de Trabalho\Classes\semestre2\IAP\project\.venv\Lib\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


   ht_form    at_form       joint_attack h2h_dominance odds_favorite result
0     good  below avg  average x average      balanced          home      A
1  average    average     average x weak      balanced          home      D
2  average    average  average x average      balanced          home      D
3     poor       poor        weak x weak      balanced          home      H
4     poor       good   weak x above avg      balanced      balanced      H
{'ht_form': ['average', 'below avg', 'excellent', 'good', 'poor'], 'at_form': ['average', 'below avg', 'excellent', 'good', 'poor'], 'joint_attack': ['above avg x above avg', 'above avg x average', 'above avg x strong', 'above avg x weak', 'average x above avg', 'average x average', 'average x strong', 'average x weak', 'strong x above avg', 'strong x average', 'strong x strong', 'strong x weak', 'weak x above avg', 'weak x average', 'weak x strong', 'weak x weak'], 'h2h_dominance': ['away_dominant', 'balanced', 'home_dominant'], 'odds_fa

Rede Bayesiana Manual - Knowledge Engineering

In [3]:
manual_edges = [
    ("ht_form", "result"),
    ("at_form", "result"),
    ("joint_attack", "result"),
    ("h2h_dominance", "result"),
    ("odds_favorite", "result"),
]

manual_structure = DiscreteBayesianNetwork(manual_edges)

print("Nós:", manual_structure.nodes())
print("Arestas:", manual_structure.edges())

Nós: ['ht_form', 'result', 'at_form', 'joint_attack', 'h2h_dominance', 'odds_favorite']
Arestas: [('ht_form', 'result'), ('at_form', 'result'), ('joint_attack', 'result'), ('h2h_dominance', 'result'), ('odds_favorite', 'result')]


Aprendizagem de Parâmetros: MLE vs Bayesiana

In [4]:
def fit_mle_model(edges, train_data):
    model = DiscreteBayesianNetwork(edges)

    estimator = MaximumLikelihoodEstimator(
        model=model,
        data=train_data,
    )

    cpds = estimator.get_parameters(n_jobs=1)
    model.add_cpds(*cpds)

    return model


def fit_bayesian_model(edges, train_data, equivalent_sample_size=10):
    model = DiscreteBayesianNetwork(edges)

    estimator = BayesianEstimator(
        model=model,
        data=train_data,
    )

    cpds = estimator.get_parameters(
        prior_type="BDeu",
        equivalent_sample_size=equivalent_sample_size,
        n_jobs=1,
    )

    model.add_cpds(*cpds)

    return model


manual_mle_model = fit_mle_model(
    manual_edges,
    train_data,
)

manual_bayes_model = fit_bayesian_model(
    manual_edges,
    train_data,
    equivalent_sample_size=10,
)

print("MLE model valid?", manual_mle_model.check_model())
print("Bayesian model valid?", manual_bayes_model.check_model())

print("\nCPD de result com MLE:")
print(manual_mle_model.get_cpds("result"))

print("\nCPD de result com estimação Bayesiana:")
print(manual_bayes_model.get_cpds("result"))

C:\Users\duart\AppData\Local\Temp\ipykernel_8328\2234224322.py:4: FutureWarning: `pgmpy.estimators.MaximumLikelihoodEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteMLE` instead.
  estimator = MaximumLikelihoodEstimator(
C:\Users\duart\AppData\Local\Temp\ipykernel_8328\2234224322.py:18: FutureWarning: `pgmpy.estimators.BayesianEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteBayesianEstimator` instead.
  estimator = BayesianEstimator(


MLE model valid? True
Bayesian model valid? True

CPD de result com MLE:
+---------------+-----+------------------------------+
| at_form       | ... | at_form(poor)                |
+---------------+-----+------------------------------+
| h2h_dominance | ... | h2h_dominance(home_dominant) |
+---------------+-----+------------------------------+
| ht_form       | ... | ht_form(poor)                |
+---------------+-----+------------------------------+
| joint_attack  | ... | joint_attack(weak x weak)    |
+---------------+-----+------------------------------+
| odds_favorite | ... | odds_favorite(home)          |
+---------------+-----+------------------------------+
| result(A)     | ... | 0.3333333333333333           |
+---------------+-----+------------------------------+
| result(D)     | ... | 0.3333333333333333           |
+---------------+-----+------------------------------+
| result(H)     | ... | 0.3333333333333333           |
+---------------+-----+------------------------

D-Separação

In [5]:
def dsep_report(model):
    pairs = [
        ("ht_form", "at_form", None),
        ("ht_form", "at_form", ["result"]),
        ("ht_form", "odds_favorite", None),
        ("joint_attack", "h2h_dominance", None),
        ("joint_attack", "h2h_dominance", ["result"]),
    ]

    for x, y, observed in pairs:
        observed_text = observed if observed is not None else []
        connected = model.is_dconnected(x, y, observed=observed_text)
        print(f"{x} e {y} | observado={observed_text}: d-connected = {connected}")


dsep_report(manual_structure)

ht_form e at_form | observado=[]: d-connected = False
ht_form e at_form | observado=['result']: d-connected = True
ht_form e odds_favorite | observado=[]: d-connected = False
joint_attack e h2h_dominance | observado=[]: d-connected = False
joint_attack e h2h_dominance | observado=['result']: d-connected = True


Inferências Probabilísticas

In [6]:
def query_result(model, evidence):
    infer = VariableElimination(model)

    q = infer.query(
        variables=["result"],
        evidence=evidence,
        show_progress=False,
    )

    result_states = q.state_names["result"]
    probs = q.values

    output = pd.DataFrame({
        "result": result_states,
        "probability": probs,
    })

    return output.sort_values("probability", ascending=False)


# 1. Probabilidade de resultado quando a equipa da casa tem boa forma e é favorita
q1 = query_result(
    manual_bayes_model,
    evidence={
        "ht_form": "good",
        "odds_favorite": "home",
    },
)
print("\nQuery 1:")
print(q1)


# 2. Probabilidade de empate quando ambas as equipas têm forma média
q2 = query_result(
    manual_bayes_model,
    evidence={
        "ht_form": "average",
        "at_form": "average",
    },
)
print("\nQuery 2:")
print(q2)


# 3. Resultado mais provável dado o histórico de confrontos diretos
q3 = query_result(
    manual_bayes_model,
    evidence={
        "h2h_dominance": "home_dominant",
    },
)
print("\nQuery 3:")
print(q3)


# 4. Exemplo mais completo
q4 = query_result(
    manual_bayes_model,
    evidence={
        "ht_form": "good",
        "at_form": "below avg",
        "joint_attack": "strong x weak",
        "odds_favorite": "home",
    },
)
print("\nQuery 4:")
print(q4)


Query 1:
  result  probability
2      H     0.495140
1      D     0.281453
0      A     0.223406

Query 2:
  result  probability
2      H     0.357099
0      A     0.340512
1      D     0.302390

Query 3:
  result  probability
2      H     0.473229
1      D     0.271760
0      A     0.255011

Query 4:
  result  probability
1      D     0.406029
0      A     0.380783
2      H     0.213187


Aprendizagem Automática da Estrutura

In [7]:
def learn_structure_hill_climb(train_data, max_indegree=3):
    hc = HillClimbSearch(train_data)

    learned_dag = hc.estimate(
        scoring_method="bic-d",
        max_indegree=max_indegree,
        max_iter=int(1e4),
    )

    learned_edges = list(learned_dag.edges())
    return learned_edges


learned_edges = learn_structure_hill_climb(train_data)

print("Arestas aprendidas automaticamente:")
for edge in learned_edges:
    print(edge)

C:\Users\duart\AppData\Local\Temp\ipykernel_8328\1716762185.py:2: FutureWarning: HillClimbSearch is deprecated and will be removed in v1.3.0. Please use
            pgmpy.causal_discovery.HillClimbSearch instead.
  hc = HillClimbSearch(train_data)
  0%|          | 6/10000 [00:00<09:16, 17.95it/s]

Arestas aprendidas automaticamente:
('at_form', 'joint_attack')
('joint_attack', 'ht_form')
('h2h_dominance', 'odds_favorite')
('h2h_dominance', 'result')
('odds_favorite', 'at_form')
('odds_favorite', 'result')


Comparação entre estrutura manual e automática

In [8]:
auto_mle_model = fit_mle_model(
    learned_edges,
    train_data,
    #state_names,
)

auto_bayes_model = fit_bayesian_model(
    learned_edges,
    train_data,
    #state_names,
    equivalent_sample_size=10,
)

print("Auto MLE model valid?", auto_mle_model.check_model())
print("Auto Bayesian model valid?", auto_bayes_model.check_model())

C:\Users\duart\AppData\Local\Temp\ipykernel_8328\2234224322.py:4: FutureWarning: `pgmpy.estimators.MaximumLikelihoodEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteMLE` instead.
  estimator = MaximumLikelihoodEstimator(


Auto MLE model valid? True
Auto Bayesian model valid? True


C:\Users\duart\AppData\Local\Temp\ipykernel_8328\2234224322.py:18: FutureWarning: `pgmpy.estimators.BayesianEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteBayesianEstimator` instead.
  estimator = BayesianEstimator(


In [9]:
manual_edge_set = set(manual_edges)
learned_edge_set = set(learned_edges)

common_edges = manual_edge_set.intersection(learned_edge_set)
only_manual = manual_edge_set - learned_edge_set
only_auto = learned_edge_set - manual_edge_set

print("Arestas em comum:")
print(common_edges)

print("\nSó na rede manual:")
print(only_manual)

print("\nSó na rede automática:")
print(only_auto)

Arestas em comum:
{('h2h_dominance', 'result'), ('odds_favorite', 'result')}

Só na rede manual:
{('joint_attack', 'result'), ('at_form', 'result'), ('ht_form', 'result')}

Só na rede automática:
{('joint_attack', 'ht_form'), ('h2h_dominance', 'odds_favorite'), ('at_form', 'joint_attack'), ('odds_favorite', 'at_form')}


In [10]:
from pathlib import Path

OUTPUT_DIR = Path("../outputs/metrics")

pd.DataFrame(manual_edges, columns=["parent", "child"]).to_csv(
    OUTPUT_DIR / "bn_manual_edges.csv",
    index=False
)

pd.DataFrame(learned_edges, columns=["parent", "child"]).to_csv(
    OUTPUT_DIR / "bn_learned_edges.csv",
    index=False
)